In [1]:
%load_ext autoreload
%autoreload 2

from paper_utils import *
    
import os
os.chdir('../..')
from sklearn.metrics import cohen_kappa_score


```
# all runs at 8 generations (this is why we have to re-run some stuff)

'8ehbkd99': 'squad-8',
'5bqecpaq': 'squad-gpt3.5',


# rerun with 8 generations for llama and deberta
# (for both of them lax entailment was better than strict)
# 171235, 171236


# let's ignore the turbo variant
# 'ogqt6jkk': 'squad-gpt4t',


# could also use older results from here (2023-10-26-long-sentences-gpt4-and-llama-entailment)
# but they are over other datapoints which is suboptimal
# for deberta lax is better than strict
# 'ooslwgb9': '70b-squad-lax-deberta',
# 'iuh4ny1v': '70b-squad-strict-deberta',
# 'fbtvpw1y': '70b-squad-lax-llama',
# 'h5vd2k6a': '70b-squad-strict-llama',
# 'rzj7w9ei': '70b-squad-lax-gpt4',
# 'e0z3555u': '70b-squad-strict-gpt4',
```

In [5]:
# only here are questions/answers stored
base_run = 'uvfbxm6d'
base_config, base_generations = restore_file(base_run, filenames=['config.yaml', 'validation_generations.pkl'])

In [2]:
runs = {
    '8ehbkd99': ['squad-gpt4', ['config.yaml', 'uncertainty_measures.pkl', 'wandb-summary.json']],
    '5bqecpaq': ['squad-gpt35', ['config.yaml', 'uncertainty_measures.pkl', 'wandb-summary.json']],
    # 'xxxxx': ['squad-llama2', ['config.yaml', 'uncertainty_measures.pkl', 'wandb-summary.json']],
    # 'xxxxx': ['squad-deberta', ['config.yaml', 'uncertainty_measures.pkl', 'wandb-summary.json']],
}

all_configs, all_results, all_perfs = {}, {}, {}
for wandb_id, (name, files) in runs.items():
    all_configs[wandb_id], all_results[wandb_id], all_perfs[wandb_id] = restore_file(wandb_id, filenames=files)

In [17]:
active_run = '8ehbkd99'

N_ANNOTATIONS = 100
START_i = 0
END_i = 100
USE_N_GENERATIONS = 8

columns = ['index', 'data_id', 'wandb_id', 'dataset', 'entailment_mode', 'human_rater', 'entailment_ranking']

In [47]:
for i in range(START_i, END_i):
    key = list(base_generations.keys())[i]
    
    result = base_generations[key]
    print(80 * f'-')
    print(f'{i} / {key} -------- New Question ------')
    print('Q:', result['question'])
    print('True answer:', result['reference']['answers']['text'])
    print('Low-Temp Model response:', result['most_likely_answer']['response'])

    # NAIVE in-order printing
    # print('------')
    # for i, response in enumerate([r[0] for r in result['responses']][:USE_N_GENERATIONS]):
    #     print(f'{i}: {response}')
    # print('------')
    # active_results = all_results[active_run]
    # print(f"semantic_ids: {active_results['semantic_ids'][i]}")

    sids = np.array(active_results['semantic_ids'][i])

    all_sids, sids_counts = np.unique(sids, return_counts=True)
    sids_by_decreasing_count = all_sids[np.argsort(sids_counts)[::-1]]
    
    # NICER printing by semantic ID
    # sort clusters by size
    for i, uniq_sid in enumerate(sids_by_decreasing_count):
        # get indices of that semantic_id
        for idx in np.where(sids == uniq_sid)[0]:
            print(colorize(f'cluster {uniq_sid}: {result["responses"][idx][0]}', i))            

    print('-----')


--------------------------------------------------------------------------------
0 / 57338007d058e614000b5bdb -------- New Question ------
Q: What was Warsaw's population in 1901?
True answer: ['711,988', '711,988', '711,988']
Low-Temp Model response: According to the 1901 census, Warsaw's population was approximately 750,000 people.
cluster 2: According to the census of 1901, Warsaw's population was 750,000.
cluster 2: According to the 1901 census, Warsaw's population was approximately 750,000 people.
cluster 2: According to the census of 1901, Warsaw's population was 750,000.
cluster 5: According to the census of 1901, Warsaw had a population of 753,011.
cluster 4: In 1901, Warsaw's population was approximately 700,000.
cluster 3: According to the All-Russia Census of 1897, Warsaw's population was approximately 750,000 people.
cluster 1: According to the census of 1901, Warsaw's population was 817,000.
cluster 0: According to the census of 1901, Warsaw's population was 756,475.
-----

array([2, 5, 4, 3, 1, 0])

In [33]:
def colorize(text, color=0):
    i2c = {i: f"\x1b[{a}" for i, a in enumerate([
    '31m', '33m', '34m', '35m', '36m', '37m', '38m', '39m'])}
    r = "\x1b[0m"

    return i2c[color] + text + r

In [ ]:


runs_data = []

for i in range(START_i, END_i):

    
    for run in runs:
        if runs[run][1][1] == 'uncertainty_measures.pkl':
            accuracy = 1 - all_results[run]['validation_is_false'][i]
        elif runs[run][1][1] == 'validation_generations.pkl':
            accuracy = result['most_likely_answer']['accuracy']
        else:
            raise
        accuracy = int(accuracy)
        dataset, metric = runs[run][0].split('-')
        run_data = [i, key, run, dataset, metric, accuracy]
        print(f'Accuracy {metric}: {accuracy}')

        if metric in WRITE_AUTOMATED_METRICS:
            runs_data.append(run_data)

        print(','.join([str(i) for i in run_data]))

    print(f'Accuracy human: ???')
    for user in users:
        runs_data.append([i, key, run, dataset, f'human_{user}', 'FILL_IN'])

In [22]:
users = ['jansen', 'sebhar', 'kunda']

# Enable this to also get gpt/llama2 truth data in the CSV
WRITE_AUTOMATED_METRICS = ['f1']

In [23]:
columns = ['index', 'data_id', 'wandb_id', 'dataset', 'truth_metric', 'truth_value']

runs_data = []

for i in range(START_i, END_i):

    key = list(all_results[active_run].keys())[i]
    
    result = all_results[active_run][key]
    print(80 * f'-')
    print(f'{i} / {key} -------- New Question ------')
    print('Q:', result['question'])
    print('True answer:', result['reference']['answers']['text'])
    print('Model response:', result['most_likely_answer']['response'])
    
    for run in runs:
        if runs[run][1][1] == 'uncertainty_measures.pkl':
            accuracy = 1 - all_results[run]['validation_is_false'][i]
        elif runs[run][1][1] == 'validation_generations.pkl':
            accuracy = result['most_likely_answer']['accuracy']
        else:
            raise
        accuracy = int(accuracy)
        dataset, metric = runs[run][0].split('-')
        run_data = [i, key, run, dataset, metric, accuracy]
        print(f'Accuracy {metric}: {accuracy}')

        if metric in WRITE_AUTOMATED_METRICS:
            runs_data.append(run_data)

        print(','.join([str(i) for i in run_data]))

    print(f'Accuracy human: ???')
    for user in users:
        runs_data.append([i, key, run, dataset, f'human_{user}', 'FILL_IN'])

--------------------------------------------------------------------------------
0 / 57338007d058e614000b5bdb -------- New Question ------
Q: What was Warsaw's population in 1901?
True answer: ['711,988', '711,988', '711,988']
Model response: According to the 1901 census, Warsaw's population was approximately 750,000 people.
Accuracy llama2: 0
0,57338007d058e614000b5bdb,uvfbxm6d,squad,llama2,0
Accuracy gpt35: 0
0,57338007d058e614000b5bdb,m3y3x605,squad,gpt35,0
Accuracy gpt4: 0
0,57338007d058e614000b5bdb,6cxqnx2u,squad,gpt4,0
Accuracy f1: 0
0,57338007d058e614000b5bdb,dcnndvny,squad,f1,0
Accuracy human: ???
--------------------------------------------------------------------------------
1 / 571cc5c45efbb31900334dde -------- New Question ------
Q: When did O2 begin to acculturate in the atmosphere?
True answer: ['2.5 billion years ago', '2.5 billion years ago', 'about 2.5 billion years ago', 'about 2.5 billion years ago', '2.5 billion years ago during the Great Oxygenation Event']
Model r

In [24]:
# [DONE] Use this to set up the CSV that you fill values in
print(pd.DataFrame(runs_data, columns=columns).sort_values(['truth_metric', 'index']).to_csv(index=False))

index,data_id,wandb_id,dataset,truth_metric,truth_value
0,57338007d058e614000b5bdb,dcnndvny,squad,f1,0
1,571cc5c45efbb31900334dde,dcnndvny,squad,f1,1
2,5733a5f54776f41900660f46,dcnndvny,squad,f1,0
3,5727dd2e4b864d1900163eba,dcnndvny,squad,f1,0
4,5733fd66d058e614000b6737,dcnndvny,squad,f1,0
5,572a064a3f37b3190047865f,dcnndvny,squad,f1,0
6,57109275b654c5140001f9a3,dcnndvny,squad,f1,0
7,572a12386aef051400155234,dcnndvny,squad,f1,0
8,57309ef18ab72b1400f9c602,dcnndvny,squad,f1,0
9,57287d4a2ca10214002da3e7,dcnndvny,squad,f1,0
10,571097baa58dae1900cd6a9b,dcnndvny,squad,f1,0
11,57300a9a04bcaa1900d77065,dcnndvny,squad,f1,0
12,572826634b864d19001645bf,dcnndvny,squad,f1,0
13,572fb059947a6a140053cb81,dcnndvny,squad,f1,0
14,5729ea263f37b319004785bd,dcnndvny,squad,f1,0
15,57264d58f1498d1400e8db7c,dcnndvny,squad,f1,0
16,5727502f708984140094dc09,dcnndvny,squad,f1,0
17,57097c8fed30961900e841f2,dcnndvny,squad,f1,0
18,57264a8cdd62a815002e808f,dcnndvny,squad,f1,1
19,57302bd0b2c2fd14005689de,dcnndvny,squad

# Load CSV and evaluate agreement

In [19]:
os.getcwd()
# os.chdir("jansen/semantic_uncertainty")

'/auto/users/jansen/semantic_uncertainty'

In [20]:
df = pd.read_csv('notebooks/paper_evals/23-11-24-accuracy-evaluation.csv', index_col=None)

if df.dataset.nunique() > 1:
    raise NotImplementedError('Eval only for one dataset for now')

def remove_incomplete(df):
    # filter out incomplete data!
    ignore = []
    for metric, mdf in df.groupby('truth_metric'):
        if (mdf.truth_value == 'FILL_IN').any():
            ignore.append(metric)
    print(f'Ignoring metrics {ignore} for now.')
    df = df[df.truth_metric.map(lambda x: x not in ignore)]
    return df

df = remove_incomplete(df)

if df.truth_value.nunique() > 2:
    raise ValueError

df

Ignoring metrics [] for now.


,index,data_id,wandb_id,dataset,truth_metric,truth_value
0,0,57338007d058e614000b5bdb,dcnndvny,squad,f1,0
1,1,571cc5c45efbb31900334dde,dcnndvny,squad,f1,1
2,2,5733a5f54776f41900660f46,dcnndvny,squad,f1,0
3,3,5727dd2e4b864d1900163eba,dcnndvny,squad,f1,0
4,4,5733fd66d058e614000b6737,dcnndvny,squad,f1,0
...,...,...,...,...,...,...
595,95,57107d73b654c5140001f91e,uvfbxm6d,squad,llama2,0
596,96,5728661e2ca10214002da2e9,uvfbxm6d,squad,llama2,0
597,97,56e17a7ccd28a01900c679a1,uvfbxm6d,squad,llama2,1
598,98,571c97e2dd7acb1400e4c121,uvfbxm6d,squad,llama2,0


In [21]:
# compute rater agreement

pdf = df.pivot(index='index', columns='truth_metric', values='truth_value')


def agreement(x, y):
    return np.mean(x == y)

for method in ['pearson', 'kendall', 'spearman', agreement, cohen_kappa_score]:
    print(f'method: {method}')
    display(pdf.corr(method=method))

method: pearson


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.000000,0.099755,0.125916,0.130695,0.135623,0.157129
gpt35,0.099755,1.000000,0.703303,0.770030,0.705290,0.714414
gpt4,0.125916,0.703303,1.000000,0.840692,0.725157,0.722856
human_jansen,0.130695,0.770030,0.840692,1.000000,0.791686,0.838257
human_sebhar,0.135623,0.705290,0.725157,0.791686,1.000000,0.761220
llama2,0.157129,0.714414,0.722856,0.838257,0.761220,1.000000


method: kendall


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.000000,0.099755,0.125916,0.130695,0.135623,0.157129
gpt35,0.099755,1.000000,0.703303,0.770030,0.705290,0.714414
gpt4,0.125916,0.703303,1.000000,0.840692,0.725157,0.722856
human_jansen,0.130695,0.770030,0.840692,1.000000,0.791686,0.838257
human_sebhar,0.135623,0.705290,0.725157,0.791686,1.000000,0.761220
llama2,0.157129,0.714414,0.722856,0.838257,0.761220,1.000000


method: spearman


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.000000,0.099755,0.125916,0.130695,0.135623,0.157129
gpt35,0.099755,1.000000,0.703303,0.770030,0.705290,0.714414
gpt4,0.125916,0.703303,1.000000,0.840692,0.725157,0.722856
human_jansen,0.130695,0.770030,0.840692,1.000000,0.791686,0.838257
human_sebhar,0.135623,0.705290,0.725157,0.791686,1.000000,0.761220
llama2,0.157129,0.714414,0.722856,0.838257,0.761220,1.000000


method: <function agreement at 0x7f2f0cdac7c0>


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.00,0.62,0.68,0.69,0.70,0.74
gpt35,0.62,1.00,0.86,0.89,0.86,0.86
gpt4,0.68,0.86,1.00,0.93,0.88,0.88
human_jansen,0.69,0.89,0.93,1.00,0.91,0.93
human_sebhar,0.70,0.86,0.88,0.91,1.00,0.90
llama2,0.74,0.86,0.88,0.93,0.90,1.00


method: <function cohen_kappa_score at 0x7f2fc9c2bf60>


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.000000,0.041856,0.059377,0.062878,0.066584,0.083862
gpt35,0.041856,1.000000,0.697363,0.761077,0.694457,0.688474
gpt4,0.059377,0.697363,1.000000,0.840474,0.724391,0.715505
human_jansen,0.062878,0.761077,0.840474,1.000000,0.791474,0.832215
human_sebhar,0.066584,0.694457,0.724391,0.791474,1.000000,0.757635
llama2,0.083862,0.688474,0.715505,0.832215,0.757635,1.000000


# Notes

* On Cohen's Cappa from Wikipedia:

> Nonetheless, magnitude guidelines have appeared in the literature. Perhaps the first was Landis and Koch,[16] who characterized values < 0 as indicating no agreement and 0–0.20 as slight, 0.21–0.40 as fair, 0.41–0.60 as moderate, 0.61–0.80 as substantial, and 0.81–1 as almost perfect agreement.

* Squad-f1 completely fails for long generations!

 
## Some thoughts on evaluation:

We are checking  if the answer matches the expected answer, not if the answer is correct!

bad_questions: 

note, that I'm not including underspecified questions 
note, that for these questions, we still just check if the model matches the 'true answer' as given by the data (even if that is wrong)

```
index, data_id, reason, explanation
11, 57300a9a04bcaa1900d77065, 'True Answer Incorrect', "Quote from Wikipedia: 'The Versailles Treaty also stipulated that Allied military forces would withdraw from the Rhineland by 1935.' And in 1936 Germany re-militarized the Rhineland"
```